# 90_pipelines / 91__full_pipeline

**Entry point for the Databricks Job.**
Runs the full ingestion → transformation → analysis pipeline in sequence.

```
favorites.json              editar favoritos (00_config/favorites.json en el repo)
concept_hierarchy.json      editar jerarquía de conceptos (00_config/ en el repo)
metrics_hierarchy.json      editar jerarquía de derived metrics (00_config/ en el repo)
      ↓
02__tickers_master              build main.config.tickers              (manual)
      ↓
03__concept_hierarchy_master    build main.config.concept_hierarchy
      ↓
04__metrics_hierarchy_master    build main.config.metrics_hierarchy
      ↓
11__fetch_sec_xbrl              SEC API → financials_raw
      ↓
12__fetch_market_data           Yahoo Finance → market_data
      ↓
21__clean_and_merge             FY rows  → MERGE into financials
21b__derive_quarterly           Q1..Q4   → MERGE into financials
21c__prune_quarterly            keep last QUARTERLY_WINDOW Qs per ticker
      ↓
22__derived_metrics             margins, FCF, YoY, leverage, valuation ratios
      ↓
31__company_analysis            validation queries
```

> **Nota:** `02__tickers_master` se ejecuta manualmente cuando editas `favorites.json`.
> Las jerarquías (`concept_hierarchy` y `metrics_hierarchy`) se reconstruyen
> automáticamente en cada run desde sus respectivos JSONs.

## 0. Pipeline parameters

Override at runtime via Databricks Job parameters:
`{"tickers_override": "AAPL,TSLA", "run_optimization": "false"}`

- `tickers_override`: lista de tickers separados por coma (omite `main.config.tickers`)
- `run_optimization`: `true` para correr OPTIMIZE + VACUUM al final

> El universo de tickers (`main.config.tickers`) se mantiene manualmente con `02__tickers_master`.

In [0]:
dbutils.widgets.text('tickers_override', '', 'tickers_override')
dbutils.widgets.text('run_optimization', 'false', 'run_optimization')
dbutils.widgets.text('rebuild_config',   'false', 'rebuild_config')

tickers_override  = dbutils.widgets.get('tickers_override')
run_optimization  = dbutils.widgets.get('run_optimization')
rebuild_config    = dbutils.widgets.get('rebuild_config')

from datetime import datetime
pipeline_start = datetime.utcnow()

print(f'tickers_override  : {tickers_override or "<empty — use main.config.tickers>"}')
print(f'run_optimization  : {run_optimization}')
print(f'rebuild_config    : {rebuild_config}')
print(f'Pipeline started  : {pipeline_start.isoformat()} UTC')

## STEP 1 / 9 — Load config

In [0]:
print('=' * 55)
print('STEP 1 / 9 — Load config')
print('=' * 55)

In [0]:
%run "/Workspace/Users/al.lopez.moreira@gmail.com/fundamentals_databricks_pj/FA PJ (Basic)/00_config/01__tickers"

In [0]:
if tickers_override.strip():
    ACTIVE_TICKERS = [t.strip().upper() for t in tickers_override.split(',') if t.strip()]
    print(f'Using override list — {len(ACTIVE_TICKERS)} tickers')
else:
    ACTIVE_TICKERS = [
        row.ticker
        for row in spark.table(f'{CATALOG}.config.tickers').select('ticker').collect()
    ]
    print(f'Loaded {len(ACTIVE_TICKERS)} tickers from {CATALOG}.config.tickers')

## STEP 2 / 9 — Concept hierarchy
Rebuilds `main.config.concept_hierarchy` from `concept_hierarchy.json`

In [0]:
print('=' * 55)
print('STEP 2 / 9 — Concept hierarchy')
print('=' * 55)

In [0]:
%run "/Workspace/Users/al.lopez.moreira@gmail.com/fundamentals_databricks_pj/FA PJ (Basic)/00_config/03__concept_hierarchy_master"

## STEP 3 / 9 — Metrics hierarchy
Rebuilds `main.config.metrics_hierarchy` from `metrics_hierarchy.json`

In [0]:
print('=' * 55)
print('STEP 3 / 9 — Metrics hierarchy')
print('=' * 55)

In [0]:
%run "/Workspace/Users/al.lopez.moreira@gmail.com/fundamentals_databricks_pj/FA PJ (Basic)/00_config/04__metrics_hierarchy_master"

## STEP 4 / 9 — Ingestion: SEC XBRL
`financials_raw` — append-only audit log (10-K + 10-Q)

In [0]:
print('=' * 55)
print('STEP 4 / 9 — SEC XBRL Ingestion')
print('=' * 55)

In [0]:
%run "/Workspace/Users/al.lopez.moreira@gmail.com/fundamentals_databricks_pj/FA PJ (Basic)/10_ingestion/11__fetch_sec_xbrl"

## STEP 5 / 9 — Ingestion: Market Data
`market_data` — year-end prices + market cap

In [0]:
print('=' * 55)
print('STEP 5 / 9 — Market Data')
print('=' * 55)

In [0]:
%run "/Workspace/Users/al.lopez.moreira@gmail.com/fundamentals_databricks_pj/FA PJ (Basic)/10_ingestion/12__fetch_market_data"

## STEP 6 / 9 — Clean & Merge (FY rows)
MERGE into `financials` — annual FY rows only

In [0]:
print('=' * 55)
print('STEP 6 / 9 — Clean & Merge (FY)')
print('=' * 55)

In [0]:
%run "/Workspace/Users/al.lopez.moreira@gmail.com/fundamentals_databricks_pj/FA PJ (Basic)/20_transformation/21__clean_and_merge"

## STEP 7 / 9 — Derive Quarterly
Standalone Q1..Q3 (with YTD fallback) + Q4 = FY − YTD_Q3; BS snapshots deduped

In [0]:
print('=' * 55)
print('STEP 7 / 9 — Derive Quarterly')
print('=' * 55)

In [0]:
%run "/Workspace/Users/al.lopez.moreira@gmail.com/fundamentals_databricks_pj/FA PJ (Basic)/20_transformation/21b__derive_quarterly"

## STEP 7b / 9 — Prune Quarterly Window
Keep only the last `QUARTERLY_WINDOW` (=12) quarters per ticker; FY rows untouched

In [0]:
print('=' * 55)
print('STEP 7b / 9 — Prune Quarterly Window')
print('=' * 55)

In [0]:
%run "/Workspace/Users/al.lopez.moreira@gmail.com/fundamentals_databricks_pj/FA PJ (Basic)/20_transformation/21c__prune_quarterly"

## STEP 8 / 9 — Derived Metrics
Compute margins, FCF, YoY growth, leverage, valuation ratios (FY only)

In [0]:
print('=' * 55)
print('STEP 8 / 9 — Derived Metrics')
print('=' * 55)

In [0]:
%run "/Workspace/Users/al.lopez.moreira@gmail.com/fundamentals_databricks_pj/FA PJ (Basic)/20_transformation/22__derived_metrics"

## STEP 9 / 9 — Analysis
Runs analysis queries — useful for validation after pipeline runs

In [0]:
print('=' * 55)
print('STEP 9 / 9 — Analysis')
print('=' * 55)

In [0]:
%run "/Workspace/Users/al.lopez.moreira@gmail.com/fundamentals_databricks_pj/FA PJ (Basic)/30_analysis/31__company_analysis"

## Pipeline summary

In [0]:
pipeline_end = datetime.utcnow()
duration     = (pipeline_end - pipeline_start).total_seconds()

print(f'\n{"="*55}')
print(f'  Pipeline completed ✓')
print(f'{"="*55}')
print(f'  Started  : {pipeline_start.isoformat()} UTC')
print(f'  Finished : {pipeline_end.isoformat()} UTC')
print(f'  Duration : {duration:.1f}s ({duration/60:.1f} min)')
print(f'  Tickers  : {len(ACTIVE_TICKERS):,}')
print()

summary_tables = [
    ('config',      'tickers'),
    ('config',      'concept_hierarchy'),
    ('config',      'metrics_hierarchy'),
    (SCHEMA,        'financials_raw'),
    (SCHEMA,        'financials'),
    (SCHEMA,        'market_data'),
    (SCHEMA,        'financials_metrics'),
]

for schema, tbl in summary_tables:
    full = f'{CATALOG}.{schema}.{tbl}'
    try:
        n = spark.table(full).count()
        print(f'  {full}: {n:,} rows')
    except Exception:
        print(f'  {full}: (not found)')

# Quarterly breakdown
print()
try:
    q_breakdown = spark.sql(f'''
        SELECT period_type, COUNT(*) AS rows, COUNT(DISTINCT ticker) AS tickers
        FROM {CATALOG}.{SCHEMA}.financials
        GROUP BY period_type
        ORDER BY period_type
    ''').collect()
    print('  financials breakdown by period_type:')
    for r in q_breakdown:
        print(f'    {r["period_type"]:>4}: {r["rows"]:>10,} rows  ({r["tickers"]:>5} tickers)')
except Exception as e:
    print(f'  (could not compute breakdown: {e})')

## Optional: optimize Delta tables
Set `run_optimization=true` in Job params to enable. Run at most once a week.

In [0]:
if run_optimization.lower() == 'true':
    print('Running OPTIMIZE + VACUUM...')
    for tbl in ['financials_raw', 'financials', 'market_data', 'financials_metrics']:
        full = f'{CATALOG}.{SCHEMA}.{tbl}'
        try:
            spark.sql(f'OPTIMIZE {full}')
            spark.sql(f'VACUUM   {full} RETAIN 168 HOURS')
            print(f'  ✓ {full}')
        except Exception as e:
            print(f'  ✗ {full}: {e}')
    print('Done.')
else:
    print('⊘ Skipping OPTIMIZE/VACUUM (set run_optimization=true to enable)')